In [2]:
import tempfile
from pathlib import Path

import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from cardprofiler.classification import split_data, build_pipeline, evaluate

df = pd.read_csv("../data/processed/df_clean.csv")
X_train, X_test, y_train, y_test = split_data(df)

# Point MLflow at the mlruns/ folder in the project root (not notebooks/mlruns)
ROOT = Path("..").resolve()

# Run data goes in a SQLite file at the project root
mlflow.set_tracking_uri(f"sqlite:///{(ROOT / 'mlflow.db').as_posix()}")

# Keep artifacts (pipeline, images) in mlruns/ at the project root, as in the brief's structure
if mlflow.get_experiment_by_name("classification_clients") is None:
    mlflow.create_experiment(
        "classification_clients",
        artifact_location=(ROOT / "mlruns").as_uri(),
    )
mlflow.set_experiment("classification_clients")

2026/10/10 23:56:47 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/10 23:56:47 INFO mlflow.store.db.utils: Updating database tables


<Experiment: artifact_location='file:///home/cdm/Coding/Projects/CardProfiler/mlruns', creation_time=1791676610588, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1791676610588, lifecycle_stage='active', name='classification_clients', tags={}, trace_location=None, workspace='default'>

In [4]:
configs = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Random Forest (tuned)": RandomForestClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM": SVC(random_state=42),
}

for name, clf in configs.items():
    pipe = build_pipeline(clf)

    with mlflow.start_run(run_name=name):
        pipe.fit(X_train, y_train)

        # hyperparameters
        mlflow.log_param("model_type", type(clf).__name__)
        mlflow.log_params(clf.get_params())

        # metrics
        metrics = evaluate(pipe, X_test, y_test)
        mlflow.log_metrics(metrics)

        # the full pipeline (scaler + model)
        mlflow.sklearn.log_model(
            pipe,
            artifact_path="pipeline",
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )
        # artifacts: confusion matrix image + classification report
        y_pred = pipe.predict(X_test)
        with tempfile.TemporaryDirectory() as tmp:
            fig, ax = plt.subplots(figsize=(6, 5))
            ConfusionMatrixDisplay.from_predictions(
                y_test, y_pred, ax=ax, cmap="Blues", xticks_rotation=45)
            ax.set_title(name)
            plt.tight_layout()
            fig.savefig(f"{tmp}/confusion_matrix.png")
            plt.close(fig)
            mlflow.log_artifact(f"{tmp}/confusion_matrix.png")

            Path(f"{tmp}/classification_report.txt").write_text(
                classification_report(y_test, y_pred))
            mlflow.log_artifact(f"{tmp}/classification_report.txt")

        print(f"{name}: F1 = {metrics['f1_score']:.4f}")

2026/10/10 23:59:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/10 23:59:47 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Random Forest: F1 = 0.9660


2026/10/10 23:59:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/10 23:59:54 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/10/10 23:59:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Random Forest (tuned): F1 = 0.9654


2026/10/10 23:59:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Decision Tree: F1 = 0.9374


2026/10/10 23:59:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/11 00:00:03 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Logistic Regression: F1 = 0.9106


2026/10/11 00:00:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/11 00:00:08 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


SVM: F1 = 0.9088
